# 02. Split Construction and Leakage Checks

This notebook converts raw split labels into a standardized three-way split and evaluates whether train, test, and OOD subsets are consistent with the intended generalization setting. Because the core experimental claim is tied to held-out-pathway generalization, leakage analysis is not optional. It determines whether test and OOD performance can be interpreted as meaningful extrapolation rather than accidental overlap.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 96.0 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM_repo
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [2]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [3]:
import anndata as ad

DATA_PATH = DEFAULT_H5AD
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"

adata = ad.read_h5ad(DATA_PATH)
print(adata)
print("obs columns:", list(adata.obs.columns))

if "dose_val" in adata.obs.columns and "dose" not in adata.obs.columns:
    adata.obs["dose"] = adata.obs["dose_val"]

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)


AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
obs columns: ['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sco

In [4]:
candidate_split_cols = [c for c in adata.obs.columns if c.startswith("split")]
candidate_split_cols


['split_ho_pathway',
 'split_tyrosine_ood',
 'split_epigenetic_ood',
 'split_cellcycle_ood',
 'split_ood_finetuning',
 'split_ho_epigenetic',
 'split_ho_epigenetic_all',
 'split_random']

In [5]:
split_reports = []
for split_col in candidate_split_cols:
    tmp = adata.obs[[split_col, "condition", "cell_type"]].copy()
    tmp["_split3"] = tmp[split_col].astype(str).map(map_split)
    tmp = tmp[tmp["_split3"].isin(["train", "test", "ood"])].copy()

    split_counts = tmp["_split3"].value_counts().to_dict()
    train_conditions = set(tmp.loc[tmp["_split3"] == "train", "condition"].astype(str))
    test_conditions = set(tmp.loc[tmp["_split3"] == "test", "condition"].astype(str))
    ood_conditions = set(tmp.loc[tmp["_split3"] == "ood", "condition"].astype(str))

    split_reports.append({
        "split_col": split_col,
        "n_train": int(split_counts.get("train", 0)),
        "n_test": int(split_counts.get("test", 0)),
        "n_ood": int(split_counts.get("ood", 0)),
        "condition_overlap_train_test": len(train_conditions & test_conditions),
        "condition_overlap_train_ood": len(train_conditions & ood_conditions),
        "condition_overlap_test_ood": len(test_conditions & ood_conditions),
    })

split_report_df = pd.DataFrame(split_reports)
split_report_df.to_csv(INTERIM_DIR / "split_column_overview.csv", index=False)
split_report_df


,split_col,n_train,n_test,n_ood,condition_overlap_train_test,condition_overlap_train_ood,condition_overlap_test_ood
0,split_ho_pathway,292283,51798,10559,188,19,19
1,split_tyrosine_ood,338824,8644,0,25,0,0
2,split_epigenetic_ood,330572,16351,0,43,0,0
3,split_cellcycle_ood,345032,5559,0,18,0,0
4,split_ood_finetuning,312405,28784,13451,179,0,0
5,split_ho_epigenetic,290889,50772,12979,180,0,0
6,split_ho_epigenetic_all,275955,49694,28991,171,0,0
7,split_random,248202,53460,52978,188,188,188


## Canonical split table

The current canonical split is expected to be `split_ho_pathway`. The next cells standardize it, save a persistent split table, and evaluate composition across cell type and condition.


In [6]:
SPLIT_COL = "split_ho_pathway"
assert SPLIT_COL in adata.obs.columns, f"Missing expected split column: {SPLIT_COL}"

split_df = adata.obs[["condition", "cell_type", "dose", SPLIT_COL] + (["pathway"] if "pathway" in adata.obs.columns else [])].copy()
split_df["_split3"] = split_df[SPLIT_COL].astype(str).map(map_split)
split_df = split_df[split_df["_split3"].isin(["train", "test", "ood"])].reset_index(names="obs_idx")
split_df.to_parquet(INTERIM_DIR / "split_table.parquet", index=False)

split_summary = split_df.groupby(["_split3", "cell_type"]).size().rename("count").reset_index()
split_summary.to_csv(INTERIM_DIR / "split_summary_by_celltype.csv", index=False)
split_summary.head()


/tmp/ipykernel_16324/2400979869.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  split_summary = split_df.groupby(["_split3", "cell_type"]).size().rename("count").reset_index()


,_split3,cell_type,count
0,ood,A549,2760
1,ood,K562,2641
2,ood,MCF7,5158
3,test,A549,12766
4,test,K562,13114


In [7]:
condition_overlap = (
    split_df.groupby(["_split3", "condition"]).size().rename("count").reset_index()
    .pivot_table(index="condition", columns="_split3", values="count", fill_value=0)
    .reset_index()
)
condition_overlap.to_csv(INTERIM_DIR / "condition_overlap_report.csv", index=False)

cell_coverage = (
    split_df.groupby(["_split3", "cell_type"]).size().rename("count").reset_index()
    .pivot_table(index="cell_type", columns="_split3", values="count", fill_value=0)
    .reset_index()
)
cell_coverage["missing_train_support"] = cell_coverage.get("train", 0) == 0
cell_coverage.to_csv(INTERIM_DIR / "cell_type_coverage_report.csv", index=False)

if "pathway" in split_df.columns:
    pathway_overlap = (
        split_df.groupby(["_split3", "pathway"]).size().rename("count").reset_index()
        .pivot_table(index="pathway", columns="_split3", values="count", fill_value=0)
        .reset_index()
    )
    pathway_overlap.to_csv(INTERIM_DIR / "pathway_overlap_report.csv", index=False)
    display(pathway_overlap.head())

display(cell_coverage)


/tmp/ipykernel_16324/1070062133.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  split_df.groupby(["_split3", "condition"]).size().rename("count").reset_index()
/tmp/ipykernel_16324/1070062133.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index="condition", columns="_split3", values="count", fill_value=0)
/tmp/ipykernel_16324/1070062133.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  split_df.groupby(["_split3", "cell_type

_split3,pathway,ood,test,train
0,Angiogenesis,776.0,2761.0,15455.0
1,Apoptosis,343.0,1332.0,7893.0
2,Cell Cycle,1286.0,4498.0,25544.0
3,Cytoskeletal Signaling,1078.0,2571.0,14121.0
4,DNA Damage,1923.0,7669.0,42885.0


_split3,cell_type,ood,test,train,missing_train_support
0,A549,2760.0,12766.0,72185.0,False
1,K562,2641.0,13114.0,73444.0,False
2,MCF7,5158.0,25918.0,146654.0,False


This notebook should be read together with the data audit. If a split leaves a cell type without train support, or if train and OOD share the same held-out biological grouping in a way that undermines the stated design, the experimental claim must be revised before training. The next notebook constructs the canonical modeling representation: train-only PCA, encoders, and control anchors.
